In [1]:
import numpy as np
import zarr
from pathlib import Path
from scipy.sparse.linalg import eigsh

ZARR_DISP = "displacements.zarr"
DESIGN_ZARR_PATH = "Design_and_Metadata.zarr"

DISP_SNAP_DATASET   = "snapshots"
DISP_POD_GROUP      = "pod_full"
DISP_POD_U_NAME     = "U"
DISP_POD_MEAN_NAME  = "mean"
DISP_POD_SVALS_NAME = "svals"

U_LATENT_TEMPLATE = "latent_displ_r{r}/coeffs"

R_MAX = 10


In [2]:
# -------------------------------------------------------
# HARD-CODE VALIDATION SIMS HERE
# -------------------------------------------------------
val_sims = np.array([210, 188, 249,   7, 584, 921, 294, 924,  85, 316, 291, 760, 889, 279, 303, 396, 580, 200,
 267, 794, 682, 310, 732, 643, 369, 832, 280, 147, 172, 196, 894, 227, 566, 733, 343, 585,
 344, 145, 352, 577, 458, 728, 916, 749, 161, 692, 660, 726, 880, 138,   5, 269, 373, 346,
 876,  41, 657, 184, 134, 683, 353, 735, 536, 306, 606, 712, 761, 724, 447, 372,  48, 748,
 859, 548, 388, 273, 900,  35, 177, 298, 785, 171, 590,   4, 314, 879, 508, 844, 699, 430,
  50, 170, 389, 919, 663, 209, 744, 164, 561, 775,  57, 131, 551, 167, 543, 203, 520, 578,
  55, 746, 521, 461, 453, 840, 847, 103, 804, 597, 814, 328, 793,  52, 852, 849, 338,  54,
 117, 831, 769, 374, 204,  31, 421, 553,  36, 702, 798, 348, 252, 313, 100, 911, 621, 274,
 270, 624, 600, 791,  78, 743, 345,  79, 162, 336, 235, 568, 616, 329, 351,  91, 807,  90,
  75, 656, 898, 531, 825, 392, 260, 829, 221, 813, 776, 511, 655, 802, 545, 677, 861, 261,
  13,  74, 185, 229, 437], dtype=np.int64)

g_design = zarr.open_group(DESIGN_ZARR_PATH, mode="r")
mgrp = g_design["Mapping_indexes_and_metadata"]
sim_index = np.asarray(mgrp["sim_index"], dtype=np.int64)

all_sims = np.unique(sim_index)
train_sims = np.setdiff1d(all_sims, val_sims)

train_mask = np.isin(sim_index, train_sims)

print("n_total sims :", len(all_sims))
print("n_train sims :", len(train_sims))
print("n_val sims   :", len(val_sims))
print("n_train snaps:", int(train_mask.sum()))
print("n_total snaps:", sim_index.shape[0])


n_total sims : 927
n_train sims : 742
n_val sims   : 185
n_train snaps: 33992
n_total snaps: 42237


In [3]:
g_disp = zarr.open_group(ZARR_DISP, mode="a")

snaps = g_disp[DISP_SNAP_DATASET]  # (M, S)
M, S_total = snaps.shape
print("snapshots shape (M,S):", (M, S_total))

train_cols = np.where(train_mask)[0]
S_train = train_cols.size
print("train snapshots:", S_train)

# Load TRAIN snapshots only
X_train = np.asarray(snaps[:, train_cols], dtype=np.float32)  # (M, S_train)

# Mean-center
mu = X_train.mean(axis=1).astype(np.float32)                  # (M,)
Xc = (X_train - mu[:, None]).astype(np.float32)               # (M, S_train)

print("Xc:", Xc.shape, Xc.dtype)


snapshots shape (M,S): (11163, 42237)
train snapshots: 33992
Xc: (11163, 33992) float32


In [4]:
# Covariance matrix
C = (Xc @ Xc.T) / max(1, (S_train - 1))
C = 0.5 * (C + C.T)  # enforce symmetry

print("C:", C.shape, C.dtype)

# Top-R_MAX eigenpairs
evals, evecs = eigsh(C, k=R_MAX, which="LM")

# Sort descending
order = np.argsort(evals)[::-1]
evals = evals[order].astype(np.float64)
U = evecs[:, order].astype(np.float32)    # (M, R_MAX)

# Singular values
svals = np.sqrt(np.maximum(evals, 0.0) * max(1, (S_train - 1))).astype(np.float64)

print("U:", U.shape)
print("svals:", svals)


C: (11163, 11163) float32
U: (11163, 10)
svals: [49861.62258096  9908.20857046  4876.92987485  2652.02295484
  1206.68867303  1014.34896622   770.26512586   678.08257593
   576.47605363   469.97667753]


In [6]:
pod_grp = g_disp.require_group(DISP_POD_GROUP)

# --- U basis ---
if DISP_POD_U_NAME in pod_grp:
    del pod_grp[DISP_POD_U_NAME]

pod_grp.create_dataset(
    name=DISP_POD_U_NAME,
    data=U,
    shape=U.shape,
    dtype=np.float32,
    chunks=(min(M, 2048), min(R_MAX, 10)),
)

# --- mean ---
if DISP_POD_MEAN_NAME in pod_grp:
    del pod_grp[DISP_POD_MEAN_NAME]

pod_grp.create_dataset(
    name=DISP_POD_MEAN_NAME,
    data=mu,
    shape=mu.shape,
    dtype=np.float32,
    chunks=(min(M, 2048),),
)

# --- singular values ---
if DISP_POD_SVALS_NAME in pod_grp:
    del pod_grp[DISP_POD_SVALS_NAME]

pod_grp.create_dataset(
    name=DISP_POD_SVALS_NAME,
    data=svals,
    shape=svals.shape,
    dtype=np.float64,
    chunks=(min(R_MAX, 10),),
)

print("✔ Overwrote POD basis (TRAIN-ONLY)")


✔ Overwrote POD basis (TRAIN-ONLY)


C:\Users\jml2414\AppData\Local\Temp\ipykernel_10420\822412699.py:7: ZarrDeprecationWarning: Use Group.create_array instead.
  pod_grp.create_dataset(
C:\Users\jml2414\AppData\Local\Temp\ipykernel_10420\822412699.py:19: ZarrDeprecationWarning: Use Group.create_array instead.
  pod_grp.create_dataset(
C:\Users\jml2414\AppData\Local\Temp\ipykernel_10420\822412699.py:31: ZarrDeprecationWarning: Use Group.create_array instead.
  pod_grp.create_dataset(


In [8]:
# Load ALL snapshots once
X_all = np.asarray(snaps[:, :], dtype=np.float32)     # (M, S)
X_allc = (X_all - mu[:, None]).astype(np.float32)

print("Projecting all snapshots:", X_allc.shape)

for r in range(1, R_MAX + 1):
    U_r = U[:, :r]                                   # (M, r)
    coeffs = (U_r.T @ X_allc).astype(np.float32)     # (r, S)

    key = U_LATENT_TEMPLATE.format(r=r)
    parent = "/".join(key.split("/")[:-1])
    leaf = key.split("/")[-1]

    grp = g_disp.require_group(parent)

    # overwrite safely (new zarr API)
    if leaf in grp:
        del grp[leaf]

    grp.create_dataset(
        name=leaf,
        data=coeffs,
        shape=coeffs.shape,
        dtype=np.float32,
        chunks=(r, min(S_total, 4096)),
    )

    print(f"✔ wrote {key} : {coeffs.shape}")


Projecting all snapshots: (11163, 42237)
✔ wrote latent_displ_r1/coeffs : (1, 42237)
✔ wrote latent_displ_r2/coeffs : (2, 42237)


C:\Users\jml2414\AppData\Local\Temp\ipykernel_10420\3700723472.py:21: ZarrDeprecationWarning: Use Group.create_array instead.
  grp.create_dataset(


✔ wrote latent_displ_r3/coeffs : (3, 42237)
✔ wrote latent_displ_r4/coeffs : (4, 42237)
✔ wrote latent_displ_r5/coeffs : (5, 42237)
✔ wrote latent_displ_r6/coeffs : (6, 42237)
✔ wrote latent_displ_r7/coeffs : (7, 42237)
✔ wrote latent_displ_r8/coeffs : (8, 42237)
✔ wrote latent_displ_r9/coeffs : (9, 42237)
✔ wrote latent_displ_r10/coeffs : (10, 42237)


In [9]:
def decode(alpha, U, mu):
    return (mu + U @ alpha).astype(np.float32)

# pick random global snapshot indices
rng = np.random.default_rng(0)
test_cols = rng.choice(S_total, size=10, replace=False)

coeffs9 = np.asarray(
    g_disp[U_LATENT_TEMPLATE.format(r=9)],
    dtype=np.float32
)

errs = []
for j in test_cols:
    x_true = X_all[:, j]
    alpha = coeffs9[:, j]
    x_hat = decode(alpha, U[:, :9], mu)

    abs_err = np.linalg.norm(x_hat - x_true)
    rel_err = abs_err / (np.linalg.norm(x_true) + 1e-12)
    errs.append((abs_err, rel_err))

errs = np.array(errs)
print("abs L2 error: mean", errs[:,0].mean(), "max", errs[:,0].max())
print("rel L2 error: mean", errs[:,1].mean(), "max", errs[:,1].max())


abs L2 error: mean 6.666546 max 17.87985
rel L2 error: mean 0.012954282 max 0.024876436
